## Miscellaneous Plots

In [ ]:
import numpy as np
import matplotlib
import pandas as pd
import os
os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
import gplately
import gplately.grids as grids
import gplately.tools as tools
import gplately.pygplates as pygplates
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import netCDF4
from scipy.interpolate import RegularGridInterpolator
from scipy import ndimage
from IPython.display import clear_output
import copy
import os, glob
import warnings
%matplotlib inline
from joblib import Parallel, delayed
import matplotlib.gridspec as gridspec
# plt.style.use('ggplot')

# common variables
extent_globe = [-180, 180, -90, 90]
earth_radius = 6371.009e3
earth_surface_area = 4.0*np.pi*earth_radius**2

# netCDF4 grid resolution
spacingX = 0.2
spacingY = 0.2
lon_grid = np.arange(extent_globe[0], extent_globe[1]+spacingX, spacingX)
lat_grid = np.arange(extent_globe[2], extent_globe[3]+spacingY, spacingY)

# reconstruction time steps and spacing
min_time = 0
max_time = 170
timestep_size = 1
reconstruction_times = np.arange(max_time, min_time-1, -timestep_size)

os.environ["DISABLE_GPLATELY_DEV_WARNING"] = "true"
plt.rcParams['font.family'] = 'Helvetica'

In [ ]:
# Don't change this: directory to input files
output_directory = "../Outputs/Videos/"
os.makedirs(output_directory, exist_ok=True)

grid_directory = "../Grids/InputGrids/"
def defineGridFiles():
    #grid_directory = input_directory+"SRGrids/"
    spreadrate_filename = grid_directory+"SpreadingRate/Alfonso2024_SPREADING_RATE_grid_{:.2f}Ma.nc"
    agegrid_filename = grid_directory+"SeafloorAge/Alfonso2024_SEAFLOOR_AGE_grid_{:.2f}Ma.nc"
    return agegrid_filename, spreadrate_filename


### Read csv files

In [ ]:
# MOR outgassing - 10Myr median filtered when saving csv
ridge_outflux_pd = pd.read_csv('../Outputs/Notebook03/csv/03_ridge_outflux.csv', index_col=0, header=[0,1])
ridge_outflux_pd = ridge_outflux_pd.reindex(index=reconstruction_times)
ridge_outflux = ridge_outflux_pd.loc[:, (['outflux'], ['min', 'mean', 'max'])].to_numpy()
ridge_outflux_min = ridge_outflux_pd['outflux']['min'].to_numpy()
ridge_outflux_mean = ridge_outflux_pd['outflux']['mean'].to_numpy()
ridge_outflux_max = ridge_outflux_pd['outflux']['max'].to_numpy()

lithosphere_influx = ridge_outflux_pd.loc[:, (['influx'], ['min', 'mean', 'max'])].to_numpy()
lithosphere_influx_min = ridge_outflux_pd['influx']['min'].to_numpy()
lithosphere_influx_mean = ridge_outflux_pd['influx']['mean'].to_numpy()
lithosphere_influx_max = ridge_outflux_pd['influx']['max'].to_numpy()

# plate influx - 10Myr median filtered
plate_influx = pd.read_csv('../Outputs/Notebook02/csv/02_plate_influx.csv', index_col=0, header=[0,1])
plate_influx = plate_influx.reindex(index=reconstruction_times)
sediments_influx    = plate_influx.loc[:, (['sediments'], ['min', 'mean', 'max'])].to_numpy()
serpentinite_influx = plate_influx.loc[:, (['serpentinite_total'], ['min', 'mean', 'max'])].to_numpy()
crust_influx        = plate_influx.loc[:, (['crust'], ['min', 'mean', 'max'])].to_numpy()
upper_plate_influx  = sediments_influx + serpentinite_influx + crust_influx
carbon_plate_influx = plate_influx.loc[:, (['serpentinite_total', 'crust', \
                                                    'lithosphere'], \
                                                   ['min', 'mean', 'max'])].to_numpy()

# subduction_outgassing - 10Myr median filtered (this depends on a 10-myr filtered output from 02_subducted_carbon)
subduction_outflux_pd = pd.read_csv('../Outputs/Notebook03/csv/03_slab_outflux.csv', index_col=0, header=[0,1])
subduction_outflux_pd = subduction_outflux_pd.reindex(index=reconstruction_times)
subduction_outflux = subduction_outflux_pd.loc[:, (['outflux_sediments', 'outflux_intrusive', \
                                                    'outflux_volcanics', 'outflux_mantlelit'], \
                                                   ['min', 'mean', 'max'])].to_numpy()

In [ ]:
carbon_components = ["Serpentinite", "Crust", "Lithosphere"]


fig = plt.figure(figsize=(5,3.5))
ax = fig.add_subplot(111, xlabel='Age (Ma)', ylabel='Rate of carbon flux (Mt C/yr)', xlim=[max_time,min_time])
# ax.set_yscale('log')


for c, component in enumerate(carbon_components):
    ax.fill_between(reconstruction_times,
                    carbon_plate_influx[:,3*c+0],
                    carbon_plate_influx[:,3*c+2],
                    color='C{}'.format(c+1), alpha=0.2)
    label = str(component)
#     if component == "Serpentinite":
#         label += ' (MOR)'
    ax.plot(reconstruction_times, carbon_plate_influx[:,3*c+1], c="C{}".format(c+1), label=label)
    
ax.fill_between(reconstruction_times,
                ridge_outflux_min,
                ridge_outflux_max,
                color='0.5', alpha=0.3)
ax.plot(reconstruction_times, ridge_outflux_mean, c='k', linewidth=2, label='MOR outflux')
    
ax.legend(prop={'size': 8})

fig.savefig(output_directory+"plate_in-outflux_comparison_sameaxes.pdf", bbox_inches='tight', dpi=300)
fig.savefig(output_directory+"plate_in-outflux_comparison_sameaxes.svg", bbox_inches='tight', dpi=300)
fig.savefig(output_directory+"plate_in-outflux_comparison_sameaxes.png", bbox_inches='tight', dpi=300)

## Diamonds

In [ ]:
diamond_age, diamond_lon, diamond_lat = np.loadtxt(
    './diamonds_age_lon_lat_country_Guiliani_2019_250-0_North_America.txt', usecols=(0,1,2), unpack=True)

mask_diamonds = diamond_lon < 0
diamond_age = diamond_age[mask_diamonds]
diamond_lon = diamond_lon[mask_diamonds]
diamond_lat = diamond_lat[mask_diamonds]

In [ ]:
# total subducted carbon
output_cdf_filename = "../Grids/smoothed_cumulative_subducted_carbon/500km/{}/mean/cumulative_subducted_carbon_{}_0.nc"

# this relies on a consistent folder structure - don't meddle!
carbon_components = ["Sediment", "Mantle"]


cumulative_subducted_carbon = []

for carbon_component in carbon_components:
    carbon_grid = grids.read_netcdf_grid(output_cdf_filename.format(carbon_component, carbon_component.lower()))
    
    carbon_grid = np.nan_to_num(carbon_grid.data)
    
    cumulative_subducted_carbon.append( carbon_grid )


In [ ]:
# total subducted carbon
proj = ccrs.NearsidePerspective(-100,40)
reconstruction_time = 0


fig = plt.figure(figsize=(15,8))
ax = fig.add_subplot(111, projection=proj)
ax.set_global()

ax.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

data = np.sum(cumulative_subducted_carbon, axis=0)*1e6
lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('plasma'))
lognorm_cmap.set_bad((1,1,1), alpha=0)
im = ax.imshow(data, extent=extent_globe, origin='lower', cmap=lognorm_cmap,
norm=mcolors.LogNorm(vmin=1e1, vmax=1e4), transform=ccrs.PlateCarree(), interpolation='nearest')

sc = ax.scatter(diamond_lon, diamond_lat, c=diamond_age, s=80, edgecolor='k', 
                cmap='Greens', transform=ccrs.PlateCarree())

# add_continents(ax, reconstruction_time, facecolor='0.7', zorder=0)
# add_coastlines(ax, reconstruction_time, facecolor='w', alpha=0.25, zorder=2)
# add_ridges(ax, reconstruction_time, facecolor='none', edgecolor='k', linewidth=1.5, zorder=2)
# add_quiver(ax, reconstruction_time, color='k', alpha=0.33, zorder=3)
# add_trenches(ax, reconstruction_time, zorder=2)

ax.coastlines(zorder=3)
ax.gridlines()
fig.colorbar(im, shrink=0.4, label='Carbon area density (t/m$^2$)', extend='max')
fig.colorbar(sc, shrink=0.4, label='Diamond age (Ma)')


In [ ]:
# total subducted carbon
proj = ccrs.NearsidePerspective(-100,40)
reconstruction_time = 0


fig = plt.figure(figsize=(15,8))
ax = fig.add_subplot(111, projection=proj)
ax.set_global()

ax.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

data = cumulative_subducted_carbon[0]*1e6
lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('plasma'))
lognorm_cmap.set_bad((1,0,1), alpha=0)
im = plt.imshow(data, extent=extent_globe, origin='lower', cmap='plasma', 
                norm=mcolors.LogNorm(vmin=1e1, vmax=1e3), transform=ccrs.PlateCarree() , interpolation='nearest')

sc = ax.scatter(diamond_lon, diamond_lat, c=diamond_age, s=80, edgecolor='k', 
                cmap='Greens', transform=ccrs.PlateCarree())

# add_continents(ax, reconstruction_time, facecolor='0.7', zorder=0)
# add_coastlines(ax, reconstruction_time, facecolor='w', alpha=0.25, zorder=2)
# add_ridges(ax, reconstruction_time, facecolor='none', edgecolor='k', linewidth=1.5, zorder=2)
# add_quiver(ax, reconstruction_time, color='k', alpha=0.33, zorder=3)
# add_trenches(ax, reconstruction_time, zorder=2)

ax.coastlines(zorder=3)
ax.gridlines()
fig.colorbar(im, shrink=0.4, label='Carbon area density (t/m$^2$)', extend='max')
fig.colorbar(sc, shrink=0.4, label='Diamond age (Ma)')


## Movies of convergence rates ONLY where continental arcs are

A set of global maps (500, 400, 300, 200, 100, 0) and the video from 540-0 in 1 my intervals showing plate boundaries, velocity arrows, subduction zones with teeth and continent outlines.

Using “plasma” colour to show convergence rates ONLY where continental arcs are, ie where subduction zones are adjacent to continental crust, limited from 0-10 cm/yr.
No MOR spreading rates and age of the ocean floor. Show carbonate platform locations through time, ie not “active” platforms but all platforms that exist at a given time


# Plate model

In [ ]:
model_dir = "./Alfonso_etal_2024_modClennettMuller/"

feature_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/"
        r"*.gpml",
    )
)

rotation_filenames = glob.glob(
    os.path.join(
        model_dir,
        "**/",
        r"*.rot",
    )
)
coastlines_filename = os.path.join(
    model_dir,
    "Coastlines",
    "Clennett__etal_2020_Coastlines.gpml",
)

static_polygons = os.path.join(
    model_dir,
    "StaticPolygons/Clennett_2020_StaticPolygons.gpml"
)

model = gplately.PlateReconstruction(
    rotation_model=rotation_filenames,
    topology_features=pygplates.FeatureCollection(
        [
            i for i in pygplates.FeaturesFunctionArgument(
                feature_filenames
            ).get_features()
            if i.get_feature_type().to_qualified_string()
            != "gpml:TopologicalSlabBoundary"
        ]
        
    ),
    static_polygons=static_polygons
)
gplot = gplately.PlotTopologies(
    plate_reconstruction=model,
    continents=coastlines_filename,
)

max_time = 170
min_time = 0
reconstruction_times = np.arange(max_time, min_time-1, -1)

# Cumulative subducted carbon through time


Load kimberlites, lamproites and carbonatites...

In [ ]:
import pandas as pd

#kimberlites = "./kimberlites_Guiliani_Pearson_2019.txt"
lamproites = "./lamproites_georoc_25Aug2023.txt"
carbonatites = "./carbonatites_Wooley_Kiarsgaard_2008.txt"

# Read deposit txt files and ignore deposits older than 1Ga
#kimberlites_pd = pd.read_table(kimberlites, names=["Longitude", "Latitude", "Age"]).iloc[1:].astype(float)
#kimberlites_pd = kimberlites_pd.loc[kimberlites_pd["Age"] <= 1000. ]
#k_lats = kimberlites_pd["Latitude"].astype(float).to_numpy()
#k_lons = kimberlites_pd["Longitude"].astype(float).to_numpy()
#k_ages = kimberlites_pd["Age"].astype(float).to_numpy()

lamproites_pd = pd.read_table(lamproites, index_col=False, names=["Longitude", "Latitude", "Age"]).iloc[1:].astype(float)
lamproites_pd = lamproites_pd.loc[lamproites_pd["Age"] <= max_time ]
l_lats = lamproites_pd["Latitude"].astype(float).to_numpy()
l_lons = lamproites_pd["Longitude"].astype(float).to_numpy()
l_ages = lamproites_pd["Age"].astype(float).to_numpy()

carbonatites_pd = pd.read_table(carbonatites, names=["Longitude", "Latitude", "Age"]).iloc[1:].replace(',', '.', regex=True).astype(float)
carbonatites_pd = carbonatites_pd.loc[carbonatites_pd["Age"] <= max_time ]
c_lats = carbonatites_pd["Latitude"].astype(float).to_numpy()
c_lons = carbonatites_pd["Longitude"].astype(float).to_numpy()
c_ages = carbonatites_pd["Age"].astype(float).to_numpy()

In [ ]:
# Kimberlites
#kimberlites_points = gplately.Points(model, k_lons, k_lats)
#for i, feature in enumerate(kimberlites_points.features):
#    feature.set_valid_time(k_ages[i], 0)
#    
#kimberlites_fc = pygplates.FeatureCollection(kimberlites_points.features)
#kimberlites_fc.write('./kimberlites_Guiliani_Pearson_2019.gpml')
    
    
# Lamproites
lamproites_points = gplately.Points(model, l_lons, l_lats)
for i, feature in enumerate(lamproites_points.features):
    feature.set_valid_time(l_ages[i], 0)
lamproites_fc = pygplates.FeatureCollection(lamproites_points.features)
lamproites_fc.write('./lamproites_georoc_25Aug2023.gpml')


# Carbonatites
carbonatites_points = gplately.Points(model, c_lons, c_lats)
for i, feature in enumerate(carbonatites_points.features):
    feature.set_valid_time(c_ages[i], 0)
    
carbonatites_fc = pygplates.FeatureCollection(carbonatites_points.features)
carbonatites_fc.write('./carbonatites_Wooley_Kiarsgaard_2008.gpml')

In [ ]:
gplots = [gplot]
models = [model]

labels = ["Alfonso et al. (2024)"]
short_label = ["Alfonso2024"]
#kimberlites = [kimberlites_points]
lamproites = [lamproites_points]
carbonatites = [carbonatites_points]

def reconstruct_deposits(i, time):
    """ Reconstruct the coordinates of deposits through time while also keeping track
    of their valid times
    """
    # RECONSTRUCT KIMBERLITES
    #reconstructed_features = models[i].reconstruct(
    #        kimberlites[i].features, time, from_time=0, anchor_plate_id=0)
    #
    #curr_klon, curr_klat = gplately.tools.extract_feature_lonlat(reconstructed_features)
    #klat_fromage = [f.get_feature().get_valid_time()[0] for f in reconstructed_features]

    # RECONSTRUCT LAMPROITES
    curr_llon, curr_llat = lamproites[i].reconstruct(time, return_array=True)
    reconstructed_features = models[i].reconstruct(
            lamproites[i].features, time, from_time=0, anchor_plate_id=0)
    curr_llon, curr_llat = gplately.tools.extract_feature_lonlat(reconstructed_features)
    llat_fromage = [f.get_feature().get_valid_time()[0] for f in reconstructed_features]
    
    # RECONSTRUCT CARBONATITES
    curr_clon, curr_clat = carbonatites[i].reconstruct(time, return_array=True)
    
    reconstructed_features = models[i].reconstruct(
            carbonatites[i].features, time, from_time=0, anchor_plate_id=0)
    
    curr_clon, curr_clat = gplately.tools.extract_feature_lonlat(reconstructed_features)
    clat_fromage = [f.get_feature().get_valid_time()[0] for f in reconstructed_features]
    #print(len(curr_llon), len(curr_llat))
    return curr_llon, curr_llat, llat_fromage, \
        curr_clon, curr_clat, clat_fromage


In [ ]:
# Obtain latitudes and longitudes of lamproites and carbonatites in parallel. Running 
# pygplates in parallel requires the 'threading' backend which is incompatible
# with reading netCDFs so this is done in a separate cell to the plotting cell. 

# We reconstruct from 0-170 so we can make use of indexing
deposit_times = np.arange(min_time, max_time+1, 1)

deposit_coords = Parallel(n_jobs=-2, verbose=1, backend='threading') \
(delayed(reconstruct_deposits) \
 (0, time) for time in deposit_times)

deposit_coords = np.array(deposit_coords, dtype=object)

curr_llon = deposit_coords[:,0]
curr_llat = deposit_coords[:,1] 
l_fromages = deposit_coords[:,2]
curr_clon = deposit_coords[:,3]
curr_clat = deposit_coords[:,4] 
c_fromages = deposit_coords[:,5] 


#all_klons = [curr_klon]
#all_klats = [curr_klat]
#all_k_fromages = [k_fromages]
all_llons = [curr_llon]
all_llats = [curr_llat]
all_l_fromages = [l_fromages]
all_clons = [curr_clon]
all_clats = [curr_clat]
all_c_fromages = [c_fromages]


### Individual plots

In [ ]:
smoothed_cumulative_grid = "../Grids/smoothed_cumulative_subducted_carbon/{}km/{}/{}/cumulative_subducted_carbon_{}_{}.nc"
    
def plot_cumulative_subducted_carbon(i, component, quantity, time, distance_km=500, save_fig=False):
    
    time_extension = 2.
    
    cumulative_grids = []
    
    
    # Read the subducted carbon grid for the current component (c) and quantity (i). 
    cumulative_carbon_grid = grids.read_netcdf_grid(
        smoothed_cumulative_grid.format(distance_km, component, quantity, component.lower(), time)
    )

    #cumulative_carbon_grid_filled = grids.fill_raster(cumulative_carbon_grid)
    cumulative_carbon_grid_filled = cumulative_carbon_grid.data
    cumulative_carbon_grid_filled = np.nan_to_num(cumulative_carbon_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_carbon_grid_filled*1e6
    cumulative_grids.append(data)
        
    #cumulative_carbon_grid_filled = grids.fill_raster(cumulative_carbon_grid)
    cumulative_carbon_grid_filled = cumulative_carbon_grid.data
    cumulative_carbon_grid_filled = np.nan_to_num(cumulative_carbon_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_carbon_grid_filled*1e6
    
    proj = ccrs.Mollweide(central_longitude=60)
    
    fig = plt.figure(figsize=(20,8))
    ax3 = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=60))

    ax3.set_global()
    
    ax3.set_title("Cumulative subducted carbon \n {} - {}Ma \n {}".format(component.replace("_", " "), time, labels[i]), fontsize=15)

    ax3.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

    lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('turbo'))
    lognorm_cmap.set_bad((1,1,1), alpha=0)

    if component == "Sediment" or component == "Crust":
        vmax = 1e3
    elif component == "Organic_Sediments":
        vmax = 1e2
    else:
        vmax = 1e9
        
    im = ax3.imshow(
        data, extent=extent_globe, origin='lower', cmap=lognorm_cmap, 
        norm=mcolors.LogNorm(vmin=1e1, vmax=vmax), alpha=0.7,

        transform=ccrs.PlateCarree(), interpolation='nearest'
    )

    gplot.time = time
    gplot.plot_continents(ax3, facecolor='w', edgecolor='None', alpha=0.46, zorder=2)

    gplot.plot_all_topological_sections(ax3, color='grey')
    gplot.plot_ridges(ax3, color='r', linewidth=1.5, zorder=2, label = "Ridges")
    gplot.plot_trenches(ax3, zorder=2, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax3, zorder=2)

    ids = []
    for feature in gplot.topologies:
        ids.append(feature.get_reconstruction_plate_id())

    for idd in np.unique(np.array(ids)):
        
        # There are some Muller2022 plate ID polygons through time 
        # that raise errors when plotted with geopandas. The conditions below
        # skip these polygons. They were manually identified.
        
        if (time == 649. or time == 650.) and str(idd) == "98046":
            continue
        elif (time in np.arange(283, 287,1)) and str(idd) == "701":
            continue
        elif time == 248. and str(idd) == "926":
            continue
        
        # If the time and current polygon is OK to plot...
        gplot.plot_plate_polygon_by_id(ax3, idd, facecolor="None", edgecolor='grey')

    ax3.gridlines(alpha=0.5)
    
    """
    # PLOT KIMBERLITES
    # Since we reconstructed kimberlites from min to max time, time will equal its index
    curr_klon, curr_klat = all_klons[i][time], all_klats[i][time]
    ax3.scatter(
        curr_klon, curr_klat,
        marker="D", c='w', edgecolor='k', s=40, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Kimberlites"
    )
    
    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(all_k_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_klon[ind])
            new_dep_lat.append(curr_klat[ind])

    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="D", c='r', edgecolor='k', s=120, 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "New Kimberlites"
        )
    """
    # RECONSTRUCT AND PLOT LAMPROITES
    # Since we reconstructed lamproites from min to max time, time will equal its index
    curr_llon, curr_llat = all_llons[i][time], all_llats[i][time]

    ax3.scatter(
        curr_llon, curr_llat,
        marker="*", c='w', edgecolor='k', s=60, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Lamproites"
    )

    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(all_l_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_llon[ind])
            new_dep_lat.append(curr_llat[ind])


    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="*", c='r', edgecolor='k', s=210, 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "New Lamproites"
        )
        
        
        
    # RECONSTRUCT AND PLOT CARBONATITES
    # Since we reconstructed lamproites from min to max time, time will equal its index
    curr_clon, curr_clat = all_clons[i][time], all_clats[i][time]
    ax3.scatter(
        curr_clon, curr_clat,
        marker="o", c='w', edgecolor='k', s=60, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Carbonatites"
    )

    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(all_c_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_clon[ind])
            new_dep_lat.append(curr_clat[ind])

    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="o", c='r', edgecolor='k', s=210, 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "New Lamproites"
        )

        
        
    plt.colorbar(im, label='Carbon area density (t/m$^2$)', extend='max', shrink=0.5)
    ax3.legend(loc="lower right", bbox_to_anchor=[0.2, -0.1])

    os.makedirs(output_directory+"cumulative_subducted_carbon", exist_ok=True)
    if save_fig:
        fig.savefig(output_directory+"cumulative_subducted_carbon/cumulative_subducted_carbon_{}_{}_{}_{}Ma.png".format(
            component, distance_km, short_label[i], time), dpi=300, bbox_inches='tight'
        )
        plt.close()


In [ ]:
component = "Mantle"

time = 165
distance_km = 300
quantity="mean"

plot_cumulative_subducted_carbon(0, component, quantity, time, distance_km)
    

In [ ]:
component = "Sediment"

time = 0
distance_km = 500
quantity="mean"

plot_cumulative_subducted_carbon(0, component, quantity, time, distance_km)
    

In [ ]:
carbon_components = ["Mantle", "Sediment"]

for i, model in enumerate(labels):
    for c, component in enumerate(carbon_components):

        plotting_times = np.arange(170, min_time-1, -timestep_size)
        distance_km = 300

        # Produce plots in a parallel routine
        #parallel_slab_storage = Parallel(n_jobs=-3, verbose=1) \
        #(delayed(plot_cumulative_subducted_carbon) \
        # (i, component, "mean", time, distance_km, save_fig=True) for time in plotting_times)

        for time in plotting_times:
            plot_cumulative_subducted_carbon(i, component, "mean", time, distance_km, save_fig=True)

In [ ]:
import moviepy as mpy

for i, model in enumerate(labels):
    for c, component in enumerate(["Mantle", "Sediment", ]):

        plotting_times = np.arange(170, min_time-1, -timestep_size)

        frame_list = []
        for reconstruction_time in plotting_times:
            frame_list.append(
                output_directory+"/cumulative_subducted_carbon/cumulative_subducted_carbon_{}_{}_{}_{}Ma.png".format(
                component, distance_km, short_label[i], reconstruction_time)
            )

        """
        clip = mpy.ImageSequenceClip(frame_list, fps=25)
        clip.write_gif(
            output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
        )

        clip = mpy.VideoFileClip(
            output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
        )

        clip.write_videofile(
            output_directory+"/cumulative_subducted_carbon_{}_{}.mp4".format(component, short_label[i]),
            codec='mpeg4'
        )
        """
        
        clip = mpy.ImageSequenceClip(frame_list, fps=30)
        clip.write_videofile(
            output_directory+"/cumulative_subducted_carbon_{}_{}km_{}.mp4".format(component, distance_km, short_label[i]), fps=30, 
            codec='mpeg4', ffmpeg_params=['-preset', 'veryslow', '-crf', '17'] )


# Plot cumulative subducted water and deposits

In [ ]:
parent_dir = "../H2O/Grids/"

# Lithosphere
lithosphere_top_dir = parent_dir+"Reservoirs/Lithosphere/top/"
lithosphere_bottom_dir = parent_dir+ "Reservoirs/Lithosphere/bottom/"

# Crust
crust_bound_dir = parent_dir+"Reservoirs/Crust/bound/"
crust_pore_dir  = parent_dir+"Reservoirs/Crust/pore/"

# Sediments
sediments_bound_dir = parent_dir+"Reservoirs/Sediment/bound/"
sediments_pore_dir = parent_dir+"Reservoirs/Sediment/pore/"


# this relies on a consistent folder structure - don't meddle!

water_components = [lithosphere_bottom_dir, #lithosphere_top_dir,
                    crust_bound_dir, #crust_pore_dir,
                    sediments_bound_dir] #sediments_pore_dir]
headers  = ['lithosphere_bottom', #'lithosphere_top',
            'crust_bound', #'crust_pore',
            'sediment_bound']#, 'sediment_pore',
            #'mantle', 'crust', 'sediment']
quantities = ["min", "mean", "max"]



smoothed_cumulative_grid = parent_dir+"/smoothed_cumulative_subducted_water/{}km/{}/{}/cumulative_subducted_water_{}_{}.nc"
    
def plot_cumulative_subducted_water(i, component, quantity, time, distance_km=500, save_fig=False):
    
    time_extension = 2.
    
    cumulative_grids = []
    
    
    # Read the subducted water grid for the current component (c) and quantity (i). 
    cumulative_water_grid = grids.read_netcdf_grid(
        smoothed_cumulative_grid.format(distance_km, component, quantity, component.lower(), time)
    )

    #cumulative_water_grid_filled = grids.fill_raster(cumulative_water_grid)
    cumulative_water_grid_filled = cumulative_water_grid.data
    cumulative_water_grid_filled = np.nan_to_num(cumulative_water_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_water_grid_filled*1e6
    cumulative_grids.append(data)
        
    #cumulative_water_grid_filled = grids.fill_raster(cumulative_water_grid)
    cumulative_water_grid_filled = cumulative_water_grid.data
    cumulative_water_grid_filled = np.nan_to_num(cumulative_water_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_water_grid_filled*1e6
    
    proj = ccrs.Mollweide(central_longitude=60)
    
    fig = plt.figure(figsize=(20,8))
    ax3 = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=-60))

    ax3.set_global()
    
    ax3.set_title("Cumulative subducted water \n {} - {}Ma \n {}".format(component.replace("_", " "), time, labels[i]), fontsize=15)

    ax3.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

    lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('turbo'))
    lognorm_cmap.set_bad((1,1,1), alpha=0)

    if component == "sediment_bound" or component == "Crust":
        vmax = 1e5
    else:
        vmax = 1e6
        
    im = ax3.imshow(
        data, extent=extent_globe, origin='lower', cmap=lognorm_cmap, 
        norm=mcolors.LogNorm(vmin=1e1, vmax=vmax), alpha=0.7,

        transform=ccrs.PlateCarree(), interpolation='nearest'
    )

    gplot.time = time
    gplot.plot_continents(ax3, facecolor='w', edgecolor='None', alpha=0.46, zorder=2)

    gplot.plot_all_topological_sections(ax3, color='grey')
    gplot.plot_ridges(ax3, color='r', linewidth=1.5, zorder=2, label = "Ridges")
    gplot.plot_trenches(ax3, zorder=2, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax3, zorder=2)

    ids = []
    for feature in gplot.topologies:
        ids.append(feature.get_reconstruction_plate_id())

    for idd in np.unique(np.array(ids)):
        
        # There are some Muller2022 plate ID polygons through time 
        # that raise errors when plotted with geopandas. The conditions below
        # skip these polygons. They were manually identified.
        
        if (time == 649. or time == 650.) and str(idd) == "98046":
            continue
        elif (time in np.arange(283, 287,1)) and str(idd) == "701":
            continue
        elif time == 248. and str(idd) == "926":
            continue
        
        # If the time and current polygon is OK to plot...
        gplot.plot_plate_polygon_by_id(ax3, idd, facecolor="None", edgecolor='grey')

    ax3.gridlines(alpha=0.5)

    # RECONSTRUCT AND PLOT LAMPROITES
    # Since we reconstructed lamproites from min to max time, time will equal its index
    curr_llon, curr_llat = all_llons[i][time], all_llats[i][time]

    ax3.scatter(
        curr_llon, curr_llat,
        marker="*", c='w', edgecolor='k', s=60, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Lamproites"
    )

    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(all_l_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_llon[ind])
            new_dep_lat.append(curr_llat[ind])


    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="*", c='r', edgecolor='k', s=210, 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "New Lamproites"
        )
        
        
        
    # RECONSTRUCT AND PLOT CARBONATITES
    # Since we reconstructed lamproites from min to max time, time will equal its index
    curr_clon, curr_clat = all_clons[i][time], all_clats[i][time]
    ax3.scatter(
        curr_clon, curr_clat,
        marker="o", c='w', edgecolor='k', s=60, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Carbonatites"
    )

    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(all_c_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_clon[ind])
            new_dep_lat.append(curr_clat[ind])

    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="o", c='r', edgecolor='k', s=210, 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "New Lamproites"
        )

        
        
    plt.colorbar(im, label='Water area density (t/m$^2$)', extend='max', shrink=0.5)
    ax3.legend(loc="lower right", bbox_to_anchor=[0.2, -0.1])

    os.makedirs(output_directory+"cumulative_subducted_water", exist_ok=True)
    if save_fig:
        fig.savefig(output_directory+"cumulative_subducted_water/cumulative_subducted_water_{}_{}_{}_{}Ma.png".format(
            component, distance_km, short_label[i], time), dpi=300, bbox_inches='tight'
        )
        plt.close()


In [ ]:
component = "mantle"

time = 0
distance_km = 500
quantity="mean"

plot_cumulative_subducted_water(0, component, quantity, time, distance_km)
    

In [ ]:
water_components = ["Mantle", "Sediment_bound"]
plt.rcParams['font.family'] = 'Helvetica'
for i, model in enumerate(labels):
    for c, component in enumerate(water_components):

        plotting_times = np.arange(170, min_time-1, -timestep_size)
        distance_km = 500

        # Produce plots in a parallel routine
        #parallel_slab_storage = Parallel(n_jobs=-3, verbose=1) \
        #(delayed(plot_cumulative_subducted_water) \
        # (i, component, "mean", time, distance_km, save_fig=True) for time in plotting_times)

        for time in plotting_times:
            plot_cumulative_subducted_water(i, component, "mean", time, distance_km, save_fig=True)

In [ ]:
import moviepy as mpy

distance_km = 500
for i, model in enumerate(labels):
    for c, component in enumerate(["Mantle", "sediment_bound", ]):

        plotting_times = np.arange(170, min_time-1, -timestep_size)

        frame_list = []
        for reconstruction_time in plotting_times:
            frame_list.append(
                output_directory+"/cumulative_subducted_water/cumulative_subducted_water_{}_{}_{}_{}Ma.png".format(
                component, distance_km, short_label[i], reconstruction_time)
            )

        """
        clip = mpy.ImageSequenceClip(frame_list, fps=25)
        clip.write_gif(
            output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
        )

        clip = mpy.VideoFileClip(
            output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
        )

        clip.write_videofile(
            output_directory+"/cumulative_subducted_carbon_{}_{}.mp4".format(component, short_label[i]),
            codec='mpeg4'
        )
        """
        
        clip = mpy.ImageSequenceClip(frame_list, fps=30)
        clip.write_videofile(
            output_directory+"/cumulative_subducted_water_{}_{}km_{}.mp4".format(component, distance_km, short_label[i]), fps=30, 
            codec='mpeg4', ffmpeg_params=['-preset', 'veryslow', '-crf', '17'] )


# Copper porphyry

In [ ]:
copper_porphyry = "./Porphyry_datasheet.csv"
copper_pd = pd.read_csv(copper_porphyry, encoding='latin1')
copper_pd["ASSIGNED_AGE_MA"] = pd.to_numeric(copper_pd["ASSIGNED_AGE_MA"], errors='coerce')
# Clean the column by replacing blank strings with NaN
copper_pd = copper_pd[~copper_pd["ORE_TONNAGE_MT"].str.contains('<', na=False)]
copper_pd["ORE_TONNAGE_MT"] = copper_pd["ORE_TONNAGE_MT"].replace(r'^\s*$', np.nan, regex=True)
# Drop rows where ORE_TONNAGE_MT is NaN 
copper_pd = copper_pd.dropna(subset=["ORE_TONNAGE_MT"])
# Ensure age column is numeric
copper_pd["ASSIGNED_AGE_MA"] = pd.to_numeric(copper_pd["ASSIGNED_AGE_MA"], errors="coerce")

# Drop rows where age is nan, negative, or zero
copper_pd = copper_pd[copper_pd["ASSIGNED_AGE_MA"] > 0]
copper_pd = copper_pd[copper_pd["ASSIGNED_AGE_MA"] <= 170]

copper_lats = copper_pd["LATITUDE"].astype(float).to_numpy()
copper_lons = copper_pd["LONGITUDE"].astype(float).to_numpy()
copper_ages = copper_pd["ASSIGNED_AGE_MA"].astype(float).to_numpy()
copper_ore_mt = copper_pd["ORE_TONNAGE_MT"].astype(float).to_numpy()

# Generate a gplately points object to reconstruct these deposits back in time!
copper_points = gplately.Points(model, copper_lons, copper_lats)
for i, feature in enumerate(copper_points.features):
    feature.set_valid_time(copper_ages[i], 0.)
    feature.set_shapefile_attribute(key="ore", value=copper_ore_mt[i])
    
copper_fc = pygplates.FeatureCollection(copper_points.features)
copper_fc.write('./copper_deposits.gpml')



In [ ]:
def reconstruct_deposits(time, metal_points):
    """ Reconstruct the coordinates of deposits through time while also keeping track
    of their valid times
    """
    # RECONSTRUCT ORES
    reconstructed_features = model.reconstruct(
            metal_points.features, time, from_time=0, anchor_plate_id=0)
    
    curr_lon, curr_lat = gplately.tools.extract_feature_lonlat(reconstructed_features)
    
    ore_sizes = []
    for i, feature in enumerate(reconstructed_features):
        feature = feature.get_feature()
        ore_size = feature.get_shapefile_attribute('ore')
        ore_sizes.append(ore_size)
        
    fromage = [f.get_feature().get_valid_time()[0] for f in reconstructed_features]
    return curr_lon, curr_lat, fromage, ore_sizes


# We reconstruct from 0-1000 so we can make use of indexing
min_time = 0
max_time = 170
deposit_times = np.arange(min_time, max_time+1, 1)

copper_coords = Parallel(n_jobs=-2, verbose=1, backend='threading') \
(delayed(reconstruct_deposits) \
 (time, copper_points) for time in deposit_times)

In [ ]:
reconstructed_copper_coords = np.array(copper_coords, dtype=object)
all_copper_lon = reconstructed_copper_coords[:,0]
all_copper_lat = reconstructed_copper_coords[:,1]
all_copper_fromage = reconstructed_copper_coords[:,2]
all_copper_sizes = reconstructed_copper_coords[:,3]

In [ ]:
def latlonticks(ax):
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False,
              linewidth=1, color='gray', alpha=0.3,)

    ax.text(0.49,-0.03, '60°E', transform=ax.transAxes)
    ax.text(0.46,-0.03, '0°', transform=ax.transAxes)
    ax.text(0.40,-0.025, '60°W', transform=ax.transAxes)
    
    gl.top_labels=False
    gl.bottom_labels=False
    return

def normalize(array, new_min, new_max):
    old_min = np.min(array)
    old_max = np.max(array)
    normalized_array = (array - old_min) / (old_max - old_min) * (new_max - new_min) + new_min
    return normalized_array

In [ ]:
grid_name = '/Volumes/Carbon_backup/CO2_Review_Bundle/CO2_review_paper/Cao_etal_2024/H2O_review_paper/H2O_Review_Bundle/Cao2024-Oct19/smoothed_cumulative_subducted_water/500km/mantle/mean/cumulative_subducted_water_mantle_{}.nc'

save_name = output_directory+"./copper/copper_plot_{}_{}_{}.{}" 
os.makedirs(output_directory+"./copper/", exist_ok=True)

def plot_map(component, quantity, time, distance_km=500, save_fig=False):
    
    # Normalize both arrays
    # Define a large size for the invisible point to set the upper limit
    invisible_point_size = 700

    all_copper_values = np.append(all_copper_sizes[time], invisible_point_size)

    time_extension = 3.
    
    # Normalize sizes to a desired range, e.g., [10, 400]
    min_size = 0
    max_size = 750
    all_copper_values_normalised = np.interp(all_copper_values, (all_copper_values.min(), all_copper_values.max()), (min_size, max_size))
    all_copper_values_normalised_visible = all_copper_values_normalised[:-1]

    region='d'
    center_lon=0
    center_lat=0
    proj = ccrs.Mollweide(central_longitude=0)
    fig, ax = plt.subplots(1,1, subplot_kw={'projection': proj}, figsize=(20, 8), dpi=150)
        

    
    # Read the subducted carbon grid for the current component (c) and quantity (i). 
    cumulative_carbon_grid = grids.read_netcdf_grid(
        smoothed_cumulative_grid.format(distance_km, component, quantity, component.lower(), time)
    )
    #cumulative_carbon_grid_filled = grids.fill_raster(cumulative_carbon_grid)
    cumulative_carbon_grid_filled = cumulative_carbon_grid.data
    cumulative_carbon_grid_filled = np.nan_to_num(cumulative_carbon_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_carbon_grid_filled*1e6

    ax.set_global()
    
    ax.set_title("Cumulative subducted carbon \n {} - {}Ma".format(component.replace("_", " "), time), fontsize=15)

    ax.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

    lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('turbo'))
    lognorm_cmap.set_bad((1,1,1), alpha=0)

    if component == "Sediment" or component == "Crust":
        vmax = 1e3
    elif component == "Organic_Sediments":
        vmax = 1e2
    else:
        vmax = 1e9
        
    im = ax.imshow(
        data, extent=extent_globe, origin='lower', cmap=lognorm_cmap, 
        norm=mcolors.LogNorm(vmin=1e1, vmax=vmax), alpha=0.7,

        transform=ccrs.PlateCarree(), interpolation='nearest'
    )

    gplot.time = time
    gplot.plot_continents(ax, facecolor='w', edgecolor='None', alpha=0.46, zorder=2)

    gplot.plot_all_topological_sections(ax, color='grey')
    gplot.plot_ridges(ax, color='r', linewidth=1.5, zorder=2, label = "Ridges")
    gplot.plot_trenches(ax, zorder=2, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax, zorder=2)

    ids = []
    for feature in gplot.topologies:
        ids.append(feature.get_reconstruction_plate_id())

    for idd in np.unique(np.array(ids)):
        
        # There are some Muller2022 plate ID polygons through time 
        # that raise errors when plotted with geopandas. The conditions below
        # skip these polygons. They were manually identified.
        
        if (time == 649. or time == 650.) and str(idd) == "98046":
            continue
        elif (time in np.arange(283, 287,1)) and str(idd) == "701":
            continue
        elif time == 248. and str(idd) == "926":
            continue
        
        # If the time and current polygon is OK to plot...
        gplot.plot_plate_polygon_by_id(ax, idd, facecolor="None", edgecolor='grey')

    ax.gridlines(alpha=0.5)


    # PLOT copper ----------------------------------------------------------------------------------
    # Since we reconstructed deposits from min to max time, time will equal its index
    curr_copper_lon, curr_copper_lat = all_copper_lon[time], all_copper_lat[time]

    # Determine the indices of deposits with nonzero copper values
    #print(len(all_mn2_values_normalised_visible), len(curr_lon))
    nonzero_copper = np.ravel(np.argwhere(np.array(all_copper_values_normalised_visible) != 0)).astype(int)

    sc = ax.scatter(
        np.array(curr_copper_lon)[[nonzero_copper]], np.array(curr_copper_lat)[[nonzero_copper]],
        marker="o", c=np.array(all_copper_fromage[time])[[nonzero_copper]].flatten(), cmap='rainbow', edgecolor='k', s=all_copper_values_normalised_visible[nonzero_copper],  label="copper",
        vmin=min_time,
        vmax=max_time,
        transform=ccrs.PlateCarree(),
        zorder=10,
    )
    
    # If a deposit has just shown up at this timestep (and it is nonzero Mn2)...
    new_dep_lon = []
    new_dep_lat = []
    for ind, dep_from_age in enumerate(np.array(all_copper_fromage[time])[[nonzero_copper]][0]):

        # Show deposit is about to show up at its birth time and then 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_copper_lon[ind])
            new_dep_lat.append(curr_copper_lat[ind])
    if new_dep_lon:
        new_copper = ax.scatter(
            new_dep_lon, new_dep_lat,
            marker="o", c='fuchsia', edgecolor='k', s=250, 
            transform=ccrs.PlateCarree(),
            zorder=11,
            label = "copper birth"
        )



    new_copper = ax.scatter([], [], label="Copper birth", marker='o', c='fuchsia', edgecolor='k', s=300)

    handles = [new_copper]
    labels = [sc.get_label() for sc in handles]
    legend = ax.legend(handles, labels, loc='center', ncols=2, labelspacing=1.3, bbox_to_anchor=(0.2, -0.35))
    ax.add_artist(legend)

    #latlonticks(ax)
    gs = gridspec.GridSpec(2,2, hspace=0.05, wspace=0.6, height_ratios=[0.96,0.04])
    ax.text(0.49,-0.03, '0°', transform=ax.transAxes)
    
    cax1 = fig.add_axes([0.38, 0.025, 0.15, 0.02])
    fig.colorbar(im,  cax=cax1, orientation='horizontal', label='Carbon area density (t/m$^2$)', extend='max')

    cax2 = fig.add_axes([0.55, 0.025, 0.25, 0.02])
    ore_cb = fig.colorbar(sc,  cax=cax2, orientation='horizontal', label='Deposit age (Ma)', extend='max')

    ticks = np.arange(0,171,10)
    tick_labels = [str(mt) for mt in ticks]
    ore_cb.set_ticks(ticks)
    ore_cb.set_ticklabels(tick_labels)

    
    # Pseudo-plots for black and white scatterplot legend - as scatterplot size is an indicator of Mn density (0 to 1)
    copper_for_legend = ax.scatter(
        all_copper_lon[0], all_copper_lat[0],
        marker="o", c=all_copper_fromage[0], alpha=0, s=normalize(all_copper_sizes[0], min_size, max_size),  label="Ore size (Mt)",
        transform=ccrs.PlateCarree(),
    )
    copper_handles, copper_labels = copper_for_legend.legend_elements("sizes", num=7)


    # Actually ensure the pseudo plots are invisible
    for h, handle in enumerate(copper_handles):
        handle.set_alpha(1.0)  # Set full alpha for legend handles

    # Generate the legends for scatterplot sizes 
    legend1 = ax.legend(copper_handles, copper_labels, title="Copper ore tonnage (MT)", ncols=3, loc='center', labelspacing=1.8, bbox_to_anchor=(0.1, -0.09))
    ax.add_artist(legend1)
    
    ax.set_global()
    
    if save_fig:
        for out_format in ["png"]:
            fig.savefig(save_name.format(component, distance_km, time, out_format), dpi=300, bbox_inches='tight'
            )
    else:
        plt.show()
    plt.close()
    return

In [ ]:
component = "Sediment"

time = 168
distance_km = 300
quantity="mean"

plot_map(component, quantity, time, distance_km)


In [ ]:
carbon_components = [#"Mantle", 
    "Sediment"]


for c, component in enumerate(carbon_components):

    plotting_times = np.arange(170, min_time-1, -timestep_size)
    distance_km = 300

    # Produce plots in a parallel routine
    #parallel_slab_storage = Parallel(n_jobs=-3, verbose=1) \
    #(delayed(plot_cumulative_subducted_carbon) \
    # (i, component, "mean", time, distance_km, save_fig=True) for time in plotting_times)

    for time in plotting_times:
        plot_map(component, "mean", time, distance_km, save_fig=True)


In [ ]:
import moviepy.editor as mpy


for c, component in enumerate([#"Mantle", 
    "Sediment"]):

    plotting_times = np.arange(170, min_time-1, -timestep_size)

    frame_list = []
    for reconstruction_time in plotting_times:
        frame_list.append(
            output_directory+"./copper/copper_plot_{}_{}_{}.png" .format(
            component, distance_km, reconstruction_time)
        )

    """
    clip = mpy.ImageSequenceClip(frame_list, fps=25)
    clip.write_gif(
        output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
    )

    clip = mpy.VideoFileClip(
        output_directory+"/cumulative_subducted_carbon_{}_{}.gif".format(component, short_label[i])
    )

    clip.write_videofile(
        output_directory+"/cumulative_subducted_carbon_{}_{}.mp4".format(component, short_label[i]),
        codec='mpeg4'
    )
    """
    
    clip = mpy.ImageSequenceClip(frame_list, fps=25)
    clip.write_videofile(
        output_directory+"/copper_{}_{}km.mp4".format(component, distance_km), fps=24, 
        codec='mpeg4', ffmpeg_params=['-preset', 'veryslow', '-crf', '17'] )

### Cumulative subducted sedimentary carbon - porphyry Cu (170-0Ma only).
Ensure cells 1-3 and 9 are run before these next cells.


In [ ]:
copper_txt = "./porphyry_gmt.txt"

copper_pd = pd.read_csv(
    copper_txt,
    delim_whitespace=True,  
    comment="#",            
    header=None,            
    names=["Longitude", "Latitude", "Cu_symbol_size", "Age"]
)


copper_pd = copper_pd.loc[copper_pd["Age"] <= max_time ]
cu_lats = copper_pd["Latitude"].astype(float).to_numpy()
cu_lons = copper_pd["Longitude"].astype(float).to_numpy()
cu_symbols = copper_pd["Cu_symbol_size"].astype(float).to_numpy()
cu_ages = copper_pd["Age"].astype(float).to_numpy()

copper_points = gplately.Points(model, cu_lons, cu_lats)
for i, feature in enumerate(copper_points.features):
    feature.set_valid_time(cu_ages[i], 0)
    feature.set_shapefile_attribute('cu_symbol', cu_symbols[i])
    
copper_fc = pygplates.FeatureCollection(copper_points.features)
copper_fc.write('./porphyry_gmt.gpml')

def reconstruct_copper(time):
    curr_clon, curr_clat = copper_points.reconstruct(time, return_array=True)
    
    reconstructed_features = model.reconstruct(
            copper_points.features, time, from_time=0, anchor_plate_id=0)
    
    curr_clon, curr_clat = gplately.tools.extract_feature_lonlat(reconstructed_features)
    clat_fromage = [f.get_feature().get_valid_time()[0] for f in reconstructed_features]
    cu_symbolsize = [f.get_feature().get_shapefile_attribute('cu_symbol') for f in reconstructed_features]
    return curr_clon, curr_clat, clat_fromage, cu_symbolsize


# We reconstruct from 0-170 so we can make use of indexing
deposit_times = np.arange(0, 170+1, 1)

deposit_coords = Parallel(n_jobs=-2, verbose=1, backend='threading') \
(delayed(reconstruct_copper) \
 (time) for time in deposit_times)

deposit_coords = np.array(deposit_coords, dtype=object)

curr_cu_lon = deposit_coords[:,0]
curr_cu_lat = deposit_coords[:,1] 
cu_fromages = deposit_coords[:,2]
cu_symbols = deposit_coords[:,3]

all_clons = [curr_cu_lon]
all_clats = [curr_cu_lat]
all_c_fromages = [cu_fromages]
all_cu_symbols = [cu_symbols]

In [ ]:
smoothed_cumulative_grid = "../Grids/smoothed_cumulative_subducted_carbon/{}km/{}/{}/cumulative_subducted_carbon_{}_{}.nc"  

def plot_cumulative_subducted_sedimentary_carbon(time, distance_km=500, save_fig=False):
    i=0
    time_extension = 3.
    component = 'Sediment'
    quantity = 'mean'
    cumulative_grids = []
    
    
    # Read the subducted carbon grid for the current component (c) and quantity (i). 
    cumulative_carbon_grid = grids.read_netcdf_grid(
        smoothed_cumulative_grid.format(distance_km, component, quantity, component.lower(), time)
    )

    #cumulative_carbon_grid_filled = grids.fill_raster(cumulative_carbon_grid)
    cumulative_carbon_grid_filled = cumulative_carbon_grid.data
    cumulative_carbon_grid_filled = np.nan_to_num(cumulative_carbon_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_carbon_grid_filled*1e6
    cumulative_grids.append(data)
        
    #cumulative_carbon_grid_filled = grids.fill_raster(cumulative_carbon_grid)
    cumulative_carbon_grid_filled = cumulative_carbon_grid.data
    cumulative_carbon_grid_filled = np.nan_to_num(cumulative_carbon_grid_filled)

    # Plot this component's cumulative contribution 
    data = cumulative_carbon_grid_filled*1e6
    
    proj = ccrs.Mollweide(central_longitude=60)
    
    fig = plt.figure(figsize=(20,8))
    ax3 = fig.add_subplot(111, projection=ccrs.Mollweide(central_longitude=30))

    ax3.set_global()
    
    gl = ax3.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False,
              linewidth=1, color='gray', alpha=0.3,)

    ax3.text(0.49,-0.03, '30°E', transform=ax3.transAxes)
    ax3.text(0.44,-0.03, '30°W', transform=ax3.transAxes)
    ax3.text(0.40,-0.025, '120°w', transform=ax3.transAxes)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    ax3.set_title("Cumulative subducted sedimentary carbon", fontsize=15)

    ax3.imshow(np.array([[0.9,0.9],[0.9,0.9]]), extent=[-180,180,-180,180], cmap='gray', vmin=0, vmax=1,
          transform=ccrs.PlateCarree(), zorder=0)

    lognorm_cmap = copy.copy(matplotlib.cm.get_cmap('turbo'))
    lognorm_cmap.set_bad((1,1,1), alpha=0)

    if component == "Sediment" or component == "Crust":
        vmax = 5e2
    elif component == "Organic_Sediments":
        vmax = 1e2
    else:
        vmax = 1e8
    
    im = ax3.imshow(
        data, extent=extent_globe, origin='lower', cmap=lognorm_cmap, 
        norm=mcolors.LogNorm(vmin=1e1, vmax=vmax), alpha=0.7,
        transform=ccrs.PlateCarree(), interpolation='nearest'
    )

    gplot.time = time
    gplot.plot_continents(ax3, facecolor='w', edgecolor='None', alpha=0.46, zorder=2)

    gplot.plot_all_topological_sections(ax3, color='grey')
    gplot.plot_ridges(ax3, color='r', linewidth=1.5, zorder=2, label = "Ridges")
    gplot.plot_trenches(ax3, zorder=2, label = "Trenches with polarity teeth")
    gplot.plot_subduction_teeth(ax3, zorder=2)

    ids = []
    for feature in gplot.topologies:
        ids.append(feature.get_reconstruction_plate_id())

    for idd in np.unique(np.array(ids)):
        
        # There are some Muller2022 plate ID polygons through time 
        # that raise errors when plotted with geopandas. The conditions below
        # skip these polygons. They were manually identified.
        
        if (time == 649. or time == 650.) and str(idd) == "98046":
            continue
        elif (time in np.arange(283, 287,1)) and str(idd) == "701":
            continue
        elif time == 248. and str(idd) == "926":
            continue
        
        # If the time and current polygon is OK to plot...
        gplot.plot_plate_polygon_by_id(ax3, idd, facecolor="None", edgecolor='grey')

    ax3.gridlines(alpha=0.5)
    

    # RECONSTRUCT AND PLOT COPPER PORPHYRY
    # Since we reconstructed lamproites from min to max time, time will equal its index
    curr_cu_lon, curr_cu_lat = all_clons[i][time], all_clats[i][time]

    GMT_radii = np.array(all_cu_symbols[i][time]) * 11
    GMT_radii = np.pi * GMT_radii **2
          
    ax3.scatter(
        curr_cu_lon, curr_cu_lat,
        marker="o", c='w', edgecolor='k', s=GMT_radii, 
        transform=ccrs.PlateCarree(),
        zorder=6,
        label = "Porphyry Cu deposit"
    )

    # If a deposit has just shown up at this timestep...
    new_dep_lon = []
    new_dep_lat = []
    new_dep_symbol_size = []
    for ind, dep_from_age in enumerate(all_c_fromages[i][time]):

        # Show deposit is about to show up 2myr before it shows up, and continue the alert 2Myr after
        # it shows up
        time_range = np.arange(dep_from_age-time_extension, dep_from_age+time_extension+1, 1)
        if int(time) in [int(t) for t in time_range]:

            # Get a list of these deposits
            new_dep_lon.append(curr_cu_lon[ind])
            new_dep_lat.append(curr_cu_lat[ind])
            new_dep_symbol_size.append(GMT_radii[ind])

    if new_dep_lon:
        ax3.scatter(
            new_dep_lon, new_dep_lat,
            marker="o", c='r', edgecolor='k', s=np.array(new_dep_symbol_size), 
            transform=ccrs.PlateCarree(),
            zorder=6,
            #label = "Porphyry Cu deposit formation"
        )
        
    ax3.text(
        0.02, 0.98, "{} Ma".format(time),
        transform=ax3.transAxes,   # use axes coordinates
        fontsize=35,
        fontweight="bold",
        va="top",                 # align text downward
        ha="left"
    )
    from matplotlib.ticker import LogFormatter
    cbar = plt.colorbar(im, label='Carbon area density (t/m$^2$)', extend='max', shrink=0.5, pad=0.02)
    ticks = [10, 100, vmax]
    
    cbar.set_ticks(ticks)
    cbar.set_ticklabels(['10$^1$', '10$^2$', '5 x $10^2$'])


    #ax3.legend(loc="lower right", bbox_to_anchor=[0.2, -0.1], markerscale=0.5)

    from matplotlib.lines import Line2D
    
    legend_elements = [
    
        # Ridges
        Line2D(
            [0], [0],
            color='r',
            linewidth=1.5,
            label='Ridges'
        ),
    
        # Trenches
        Line2D(
            [0], [0],
            color='k',
            linewidth=1.5,
            label='Trenches with polarity teeth'
        ),
        
        # Porphyry Cu deposit (circle)
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor='w',
            markeredgecolor='k',
            markersize=10,
            linestyle='None',
            label='Porphyry Cu deposit'
        ),
    
        # New deposit formation
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            markerfacecolor='r',
            markeredgecolor='k',
            markersize=10,
            linestyle='None',
            label='Porphyry Cu deposit formation'
        ),
    ]

    ax3.legend(handles=legend_elements, loc="lower right", bbox_to_anchor=[0.2, -0.1],)

    os.makedirs(output_directory+"cumulative_subducted_sedimentary_carbon_cu", exist_ok=True)
    if save_fig:
        fig.savefig(output_directory+"cumulative_subducted_sedimentary_carbon_cu/cumulative_subducted_sedimentary_carbon_cu_{}Ma.png".format(
            time), dpi=300, bbox_inches='tight'
        )
        plt.close()


In [ ]:

time = 0
distance_km = 300


plot_cumulative_subducted_sedimentary_carbon(time, distance_km)
    

In [ ]:
plotting_times = np.arange(170, min_time-1, -timestep_size)

# Produce plots in a parallel routine
#parallel_slab_storage = Parallel(n_jobs=-3, verbose=1) \
#(delayed(plot_cumulative_subducted_sedimentary_carbon) \
# (time, distance_km, save_fig=True) for time in plotting_times)

for time in plotting_times:
    plot_cumulative_subducted_sedimentary_carbon(time, distance_km, save_fig=True)


In [ ]:
import moviepy.editor as mpy


plotting_times = np.arange(170, min_time-1, -timestep_size)

frame_list = []
for reconstruction_time in plotting_times:
    frame_list.append(
        output_directory+"./cumulative_subducted_sedimentary_carbon_cu/cumulative_subducted_sedimentary_carbon_cu_{}Ma.png" .format(
        reconstruction_time)
    )

clip = mpy.ImageSequenceClip(frame_list, fps=25)
clip.write_videofile(
    output_directory+"/cumulative_subducted_sedimentary_carbon_cu_{}km.mp4".format(distance_km), fps=24, 
    codec='mpeg4', ffmpeg_params=['-preset', 'veryslow', '-crf', '17'] )